# bottleneck-latent-projection — faded example 2: Complete the final latent projection (no activation)

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `bottleneck-latent-projection`. Running the beacon reports progress on the `Generative: Bottleneck latent projection` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Generative: Bottleneck latent projection` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`bottleneck-latent-projection`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "bottleneck-latent-projection"
DD_SUBTOPIC = "Generative: Bottleneck latent projection"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The encoder bottleneck ends with `Linear(hidden, latent)` applied to the post-ReLU hidden vector - and critically with **no** activation on its output, so the latent code can take any real value. The weight is stored `(latent, hidden)`, so the projection is `h @ W2.T + b2`.

## Faded exercise 2

Implement `encode(x, W1, b1, W2, b2)`. The flatten, first Linear, and ReLU are given. Complete the **final latent projection**: apply the second Linear to `h` with weight `W2` of shape `(latent, hidden)` and bias `b2`, applying NO activation function.

**Fill in:** Projects the hidden vector to the latent code via the second Linear (h @ W2.T + b2) with no activation.

In [ ]:
def encode(x, W1, b1, W2, b2):
    flat = rearrange(x, 'b c h w -> b (c h w)')
    h = t.relu(flat @ W1.T + b1)
    z = None  # TODO: Project the hidden vector to the latent code via the second Linear (h @ W2.T + b2) with no activation.
    return z

t.manual_seed(0)
B, C, H, W = 6, 4, 8, 8
in_f, hidden, latent = C * H * W, 64, 5
x = t.randn(B, C, H, W)
W1 = t.randn(hidden, in_f); b1 = t.randn(hidden)
W2 = t.randn(latent, hidden); b2 = t.randn(latent)
z = encode(x, W1, b1, W2, b2)
print('latent shape:', tuple(z.shape))


def _test():
    t.manual_seed(0)
    B, C, H, W = 6, 4, 8, 8
    in_f, hidden, latent = C * H * W, 64, 5
    x = t.randn(B, C, H, W)
    W1 = t.randn(hidden, in_f); b1 = t.randn(hidden)
    W2 = t.randn(latent, hidden); b2 = t.randn(latent)
    z = encode(x, W1, b1, W2, b2)
    assert z.shape == (B, latent), z.shape
    flat = x.reshape(B, in_f)
    h = t.relu(flat @ W1.T + b1)
    z_ref = h @ W2.T + b2
    assert t.allclose(z, z_ref, atol=1e-5), 'latent mismatch'
    # no activation: with these weights some latent entries must be negative
    assert (z_ref < 0).any(), 'expected unbounded (some negative) latent values'


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def encode(x, W1, b1, W2, b2):
    flat = rearrange(x, 'b c h w -> b (c h w)')
    h = t.relu(flat @ W1.T + b1)
    z = h @ W2.T + b2
    return z

t.manual_seed(0)
B, C, H, W = 6, 4, 8, 8
in_f, hidden, latent = C * H * W, 64, 5
x = t.randn(B, C, H, W)
W1 = t.randn(hidden, in_f); b1 = t.randn(hidden)
W2 = t.randn(latent, hidden); b2 = t.randn(latent)
z = encode(x, W1, b1, W2, b2)
print('latent shape:', tuple(z.shape))
```
</details>